# Day 2 - Topic 2: Lambda, map, filter, reduce

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- **Lambda** is a small, anonymous (nameless) one-line function
- **map()** applies a function to every item of an iterable
- **filter()** keeps only the items where a function returns `True`
- **reduce()** repeatedly combines items into a single final value (needs `functools` import)
- Why needed?
  - Quick throwaway logic without formally defining a function
  - These are the "functional programming" trio - transform, select, aggregate
- Where used?
  - `df["col"].apply(lambda x: ...)` - one of the most typed lines in all of Pandas
  - Sorting with custom keys: `sorted(data, key=lambda x: x["salary"])`
  - Interviewers use these to test functional-style thinking

## 2. Real-Life Analogy

- **Lambda** = a sticky note instruction ("multiply by 2") - quick, unnamed, used once, thrown away; versus a `def` function which is a laminated, titled instruction card kept in a binder
- Imagine a conveyor belt of items in a factory:
  - **map** = a stamping machine that transforms EVERY item passing through (all items come out changed)
  - **filter** = a quality-control gate that only lets qualifying items pass (fewer items come out, unchanged)
  - **reduce** = a compactor at the end that crushes everything into ONE final block (one value comes out)

> **Trick to remember:** map = "change all", filter = "choose some", reduce = "combine into one".

## 3. Explanation

- `lambda arguments: expression`
  - Single expression only - no statements, no multiple lines, the expression's value is auto-returned
- `map(function, iterable)` - returns a lazy map object; wrap in `list()` to see results
- `filter(function, iterable)` - the function must return `True`/`False`; keeps items where result is `True`
- `reduce(function, iterable)` - the function takes TWO arguments (accumulated result, next item); from `functools`
- All three accept either a lambda or a regular named function

> **Trick to remember:** map and filter return lazy objects (like generators) - nothing happens until you consume them with `list()` or a loop.

## 4. Syntax

```python
# Lambda
lambda x: x * 2
lambda x, y: x + y
lambda x: "even" if x % 2 == 0 else "odd"   # inline if/else allowed

# map / filter
map(function, iterable)
filter(function, iterable)

# reduce (requires import)
from functools import reduce
reduce(function, iterable)
reduce(function, iterable, initial_value)
```

- `lambda` - keyword to create the anonymous function
- `x` (before `:`) - parameter(s)
- expression (after `:`) - computed and returned automatically

In [ ]:
double = lambda x: x * 2      # for demo only; naming a lambda is actually discouraged (PEP 8)
print(double(5))

nums = [1, 2, 3, 4, 5]
print(list(map(lambda x: x * x, nums)))
print(list(filter(lambda x: x % 2 == 0, nums)))

from functools import reduce
print(reduce(lambda a, b: a + b, nums))    # ((((1+2)+3)+4)+5) = 15


## 5. Examples

### Basic Example

In [ ]:
# Basic: same logic as def vs lambda
def square_def(x):
    return x * x

square_lam = lambda x: x * x

print(square_def(4), square_lam(4))   # identical behavior


### Intermediate Example

In [ ]:
# Intermediate: map + filter chained, and sorting with a lambda key
prices = [120, 45, 300, 80, 15]

# Add 18% tax to every price, then keep only those above 100
taxed = list(map(lambda p: round(p * 1.18, 2), prices))
expensive = list(filter(lambda p: p > 100, taxed))
print("Taxed:", taxed)
print("Expensive:", expensive)

# Sorting a list of dicts by a field - the #1 practical lambda use
employees = [
    {"name": "Asha", "salary": 52000},
    {"name": "Vikram", "salary": 68000},
    {"name": "Neha", "salary": 49000},
]
by_salary = sorted(employees, key=lambda e: e["salary"], reverse=True)
print("Top earner:", by_salary[0]["name"])


- `map` transformed every price (tax), `filter` then selected a subset - transform-then-select is a standard pipeline pattern
- `sorted(..., key=lambda e: e["salary"])` tells sorted WHAT to compare - the lambda extracts the comparison value from each item
- `key=lambda ...` in `sorted()`/`max()`/`min()` is the single most common lambda pattern in real code and interviews

### Real-World Example

In [ ]:
# Real-world: mini feature-engineering pipeline on raw records
from functools import reduce

transactions = [
    {"id": 1, "amount": 2500, "status": "success"},
    {"id": 2, "amount": 150,  "status": "failed"},
    {"id": 3, "amount": 4200, "status": "success"},
    {"id": 4, "amount": 900,  "status": "success"},
]

# Step 1 (filter): keep successful transactions only
successful = filter(lambda t: t["status"] == "success", transactions)

# Step 2 (map): extract just the amounts
amounts = map(lambda t: t["amount"], successful)

# Step 3 (reduce): total revenue
total_revenue = reduce(lambda a, b: a + b, amounts)

print("Total successful revenue:", total_revenue)


- filter -> map -> reduce reads as: select rows -> extract column -> aggregate - which is EXACTLY what SQL's `WHERE -> SELECT -> SUM` and Pandas' filter/column-select/`.sum()` do
- Because map/filter are lazy, no intermediate lists were built - the pipeline processes items on demand
- Honest best-practice note: in real code `sum(t["amount"] for t in transactions if t["status"] == "success")` does the same thing more readably - but interviewers still expect you to know the functional trio

## 6. Internal Working

- A lambda creates a normal function object - identical machinery to `def`, except it has no name (`__name__` is `'<lambda>'`) and is limited to one expression
- `map` and `filter` return **lazy iterator objects** - they hold a reference to the function and iterable, computing one item at a time when consumed (same lazy idea as generator expressions from Day 1)
- `reduce` keeps an internal accumulator: it calls `func(acc, item)` for each item, feeding each result back in as the new accumulator

> **Trick to remember:** lambda is just a def with no name and one expression - nothing magical, same function object underneath.

In [ ]:
f = lambda x: x + 1
def g(x):
    return x + 1

print(type(f), type(g))          # both <class 'function'>
print(f.__name__, g.__name__)    # '<lambda>' vs 'g'


## 7. Time and Space Complexity

- `map`/`filter` over n items: O(n) time; O(1) extra space while lazy (O(n) only if you materialize with `list()`)
- `reduce` over n items: O(n) time, O(1) space (just the accumulator)
- Same complexity as the equivalent loop or comprehension - the difference is style and laziness, not big-O

## 8. Common Mistakes

- Printing `map(...)`/`filter(...)` directly and being confused by `<map object ...>` - wrap in `list()` to see values
- Trying to consume a map/filter object twice - like generators, they are exhausted after one pass
- Cramming complex multi-branch logic into a lambda - unreadable; use `def` when logic grows
- Forgetting `from functools import reduce` (it was moved out of built-ins in Python 3)
- Assigning lambdas to names everywhere (`f = lambda x: ...`) - PEP 8 says use `def` for named functions

In [ ]:
m = map(lambda x: x * 2, [1, 2, 3])
print(list(m))   # [2, 4, 6]
print(list(m))   # [] - exhausted, same one-shot behavior as generators


## 9. Best Practices

- Use lambdas only for short, simple, one-off logic (especially as `key=` arguments)
- If a lambda needs a name or spans complex logic, write a `def` instead
- Prefer comprehensions over map/filter for readability when building lists: `[x*2 for x in nums]` over `list(map(lambda x: x*2, nums))`
- Prefer built-in `sum()` over `reduce` for addition - reserve `reduce` for genuinely custom accumulation
- In Pandas, prefer vectorized operations over `.apply(lambda ...)` when possible - lambda apply is a fallback, not a default (full detail in Day 4)

## 10. Interview Questions

**Beginner**
- Q: What is a lambda function?
  A: An anonymous, single-expression function defined with the `lambda` keyword - the expression's result is returned automatically.
- Q: What is the difference between map and filter?
  A: `map` transforms every item using the function; `filter` selects only the items for which the function returns `True`.

**Intermediate**
- Q: Why does `print(map(f, data))` not show the results?
  A: `map` returns a lazy iterator object, not a list - values are computed only when consumed (e.g. via `list()` or a loop).
- Q: What are the limitations of lambda compared to def?
  A: A lambda can contain only a single expression - no statements (no assignments, loops, try/except), no annotations, and no real name for debugging/tracebacks.

**Advanced**
- Q: How does reduce work internally, and what does the optional third argument do?
  A: It maintains an accumulator, calling `func(accumulator, next_item)` across the iterable, using each result as the new accumulator. The optional initial value seeds the accumulator (and protects against empty-iterable errors).
- Q: When would you choose a comprehension over map/filter, and why?
  A: When readability matters (most of the time) - comprehensions express transform+filter in one familiar syntax and avoid lambda noise. map/filter shine mainly when you already have a named function to apply, e.g. `map(str.strip, lines)`.

## 11. Practice Problems

**Easy**
1. Use `map` with a lambda to convert a list of temperatures from Celsius to Fahrenheit.
2. Use `filter` with a lambda to keep only names longer than 4 characters from a list.

**Medium**
3. Given a list of dicts with "name" and "score", use `sorted` with a lambda key to get the top-3 scorers.
4. Chain map and filter: from a list of prices, first apply a 10% discount, then keep only prices under 500.

**Hard**
5. Use `reduce` to find the maximum value in a list WITHOUT using `max()`, then explain in a comment how the accumulator evolves step by step for the input `[3, 7, 2, 9, 4]`.

## 12. Revision Summary

- `lambda args: expression` - anonymous one-expression function, auto-returns
- map = change all, filter = choose some, reduce = combine into one
- map/filter return lazy, one-shot iterators - wrap in `list()` to materialize
- `reduce` needs `from functools import reduce`; takes `func(acc, item)`
- The #1 practical lambda use: `key=lambda x: ...` in sorted/max/min
- Comprehensions usually beat map/filter for readability - know both, prefer comprehensions
- All three are O(n) - same as loops, different style

> **Next topic (Day 2 continues):** Closures + Decorators